# Task: GNN training on synthetic SPV simulations: adjacency, cell state and property matrices

We will be developing a graph neural network (GNN)-based model capable of inferring mechanistic rules and uncovering the principles driving DPAC aggregation. To facilitate this, the GNN will initially be trained using synthetic Self-Propelled Voronoi (SPV) simulations, serving as placeholder data while the deep learning infrastructure is optimized. The GNN will be validated by its ability to, first, recover the physical mechanisms embedded in the SPV model, then subsequently applied to DPAC data to explore the impacts of initial thickness and cell density.

### GNN training

In [131]:
import os
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GINEConv
from torch_geometric.utils import from_networkx

from torch.optim import AdamW, lr_scheduler
from sklearn.model_selection import KFold

##########################################################################
# Custom collate function for standard DataLoader
##########################################################################

def pyg_collate_fn(batch_list):
    return Batch.from_data_list(batch_list)

##########################################################################
# 1. Parameter Parsing
##########################################################################

def parse_parameters(param_path):
    params = {}
    print(f"Parsing parameters from {param_path}")
    with open(param_path, "r") as f:
        exec(f.read(), {}, params)
    if not isinstance(params["W"], np.ndarray):
        params["W"] = np.array(params["W"])
    print(f"Successfully parsed parameters: {params}")
    return params

##########################################################################
# 2. Data Loading with Caching
##########################################################################

def load_matrices(directory, timepoint):
    cache_key = (directory, timepoint)
    if not hasattr(load_matrices, 'cache'):
        load_matrices.cache = {}
    if cache_key not in load_matrices.cache:
        print(f"Loading matrices for timepoint {timepoint} from {directory}")
        graph_mat = np.load(os.path.join(directory, f"{timepoint}_graph_mat.npy"))
        properties_mat = np.load(os.path.join(directory, f"{timepoint}_properties_mat.npy"))
        state_mat = np.load(os.path.join(directory, f"{timepoint}_state_mat.npy"))
        load_matrices.cache[cache_key] = (graph_mat, properties_mat, state_mat)
    return load_matrices.cache[cache_key]

##########################################################################
# 3. GCA Initialization
##########################################################################

def initialize_gca(graph_mat, properties_mat, state_mat, params):
    """
    Create a graph (GCA) with node properties, state, edges, etc.
    """
    print("Initializing GCA graph...")
    g = nx.from_numpy_array(graph_mat, create_using=nx.Graph)
    
    if np.any(properties_mat[:, 0] < 0) or np.any(properties_mat[:, 1] < 0):
        raise ValueError("Area and perimeter must be non-negative")

    for i, row in enumerate(properties_mat):
        area      = row[0]
        perimeter = row[1]
        cell_type = np.argmax(state_mat[i])
        g.nodes[i]["state"] = state_mat[i]
        g.nodes[i].update({
            "area": max(area, 0),
            "perimeter": max(perimeter, 0),
            "motility": params["v0"][cell_type],
            "persistence": params["Dr"],
            "kappa_A": params["kappa_A"],
            "kappa_P": params["kappa_P"],
            "A0": params["A0"][cell_type],
            "P0": params["P0"][cell_type],
        })

    # Edge props
    for u, v in g.edges():
        type_u = np.argmax(g.nodes[u]["state"])
        type_v = np.argmax(g.nodes[v]["state"])
        adhesion = params["W"][type_u][type_v]
        g.edges[u, v].update({
            "adhesion": adhesion,
            "repulsion_radius": params["a"], 
            "repulsion_coefficient": params["k"],
        })

    g.graph["adj_matrix"] = graph_mat
    return g

##########################################################################
# 4. GraphPredictor Model
##########################################################################

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_mean_pool

class DeepGraphPredictor(nn.Module):
    """
    A deeper GNN that tries to capture both local and global structure.
    1) 4 GINEConv layers for more local context
    2) A global mean pooling after the second conv, merged back into each node embedding 
       for global awareness
    3) Outputs state_pred, area_pred, perim_pred, adj_pred
    """
    def __init__(self, node_dim, edge_dim, hidden_dim, num_cell_types):
        super().__init__()

        # Initial node feature encoder
        self.node_encoder = nn.Linear(node_dim, hidden_dim)

        # MLP used by GINEConv for neighbor aggregation
        self.mlp1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.mlp2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.mlp3 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.mlp4 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # 4 GINEConv layers
        self.conv1 = GINEConv(nn=self.mlp1, edge_dim=edge_dim)
        self.conv2 = GINEConv(nn=self.mlp2, edge_dim=edge_dim)
        self.conv3 = GINEConv(nn=self.mlp3, edge_dim=edge_dim)
        self.conv4 = GINEConv(nn=self.mlp4, edge_dim=edge_dim)

        # Decoders
        self.state_decoder = nn.Linear(hidden_dim, num_cell_types)
        self.area_decoder  = nn.Linear(hidden_dim, 1)
        self.perim_decoder = nn.Linear(hidden_dim, 1)
        self.adj_decoder   = nn.Linear(2 * hidden_dim, 1)

    def forward(self, x, edge_index, edge_attr, batch=None):
        """
        batch is optional, but if provided it helps us do global pooling
        in a multi-graph scenario. If not provided, we'll treat the entire
        input as a single graph (use your own approach).
        """
        # 1) Encode node features
        x = self.node_encoder(x)  # shape [num_nodes, hidden_dim]

        # 2) GINEConv #1
        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)

        # 3) GINEConv #2
        x = self.conv2(x, edge_index, edge_attr)
        x = F.relu(x)

        # 4) Global readout:
        #    If 'batch' is None, we treat the entire graph as one component.
        #    If 'batch' is not None, we do mean_pool across each graph in the batch.
        if batch is None:
            # single-graph scenario:
            global_vec = x.mean(dim=0, keepdim=True)  # shape [1, hidden_dim]
            # broadcast back to all nodes:
            x = x + global_vec
        else:
            # multi-graph scenario:
            global_vec = global_mean_pool(x, batch)  # shape [num_graphs, hidden_dim]
            x = x + global_vec[batch]  # shape [num_nodes, hidden_dim]
        
        # 5) GINEConv #3
        x = self.conv3(x, edge_index, edge_attr)
        x = F.relu(x)

        # 6) GINEConv #4
        x = self.conv4(x, edge_index, edge_attr)
        x = F.relu(x)

        # Now we do decoders
        state_pred = F.log_softmax(self.state_decoder(x), dim=-1)  # [num_nodes, num_cell_types]
        area_pred  = self.area_decoder(x)                          # [num_nodes, 1]
        perim_pred = self.perim_decoder(x)                         # [num_nodes, 1]

        # adjacency
        row, col = edge_index
        x_u = x[row] 
        x_v = x[col] 
        edge_repr = torch.cat([x_u, x_v], dim=1)  # [num_edges, 2*hidden_dim]
        adj_pred  = self.adj_decoder(edge_repr)   # [num_edges, 1]

        return state_pred, area_pred, perim_pred, adj_pred


##########################################################################
# 5. Distance-based Weighted Loss
##########################################################################

def distance_based_adj_loss(adj_pred, next_adj, data, alpha=1.0):
    if getattr(data, 'pos', None) is None:
        return F.binary_cross_entropy_with_logits(adj_pred.squeeze(), next_adj.float())

    row, col = data.edge_index
    node_positions = data.pos
    adj_pred = adj_pred.squeeze()
    target = next_adj.float().squeeze()

    pos_row = node_positions[row]
    pos_col = node_positions[col]
    dist = torch.norm(pos_row - pos_col, dim=1)

    weights = torch.exp(-alpha * dist)
    loss = F.binary_cross_entropy_with_logits(adj_pred, target, weight=weights)
    return loss

def distance_based_node_prop_loss(pred_prop, target_prop, data, alpha=1.0):
    if getattr(data, 'pos', None) is None:
        return F.smooth_l1_loss(pred_prop, target_prop)

    node_positions = data.pos
    num_nodes = node_positions.shape[0]
    dist_weight = torch.zeros(num_nodes, dtype=pred_prop.dtype, device=pred_prop.device)

    for i in range(num_nodes):
        dist_ij = torch.norm(node_positions - node_positions[i], dim=1)
        w_ij = torch.exp(-alpha * dist_ij)
        dist_weight[i] = w_ij.mean()

    errors = F.smooth_l1_loss(pred_prop, target_prop, reduction='none').squeeze()
    weighted_error = errors * dist_weight
    return weighted_error.mean()

##########################################################################
# 5. Training & Validation
##########################################################################

def train_epoch(model, data_list, optimizer, device, num_cell_types, alpha=1.0):
    model.train()
    total_loss = 0
    
    loader = DataLoader(data_list, batch_size=1, shuffle=True, collate_fn=pyg_collate_fn)
    criterion_state = nn.KLDivLoss(reduction='batchmean')
    
    for batch_idx, batch in enumerate(loader):
        optimizer.zero_grad()
        
        batch = batch.to(device)
        
        state_pred, area_pred, perim_pred, adj_pred = model(batch.x, batch.edge_index, batch.edge_attr)

        loss_adj   = distance_based_adj_loss(adj_pred, batch.next_adj, batch, alpha=alpha)
        loss_area  = distance_based_node_prop_loss(area_pred, batch.next_area.unsqueeze(-1), batch, alpha=alpha)
        loss_perim = distance_based_node_prop_loss(perim_pred, batch.next_perim.unsqueeze(-1), batch, alpha=alpha)
        loss_state = criterion_state(state_pred, batch.next_state)

        total_batch_loss = loss_adj + loss_area + loss_perim + loss_state

        l2_reg = 1e-4 * sum(p.pow(2.0).sum() for p in model.parameters())
        total_batch_loss += l2_reg

        total_batch_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += total_batch_loss.item()
        
        if (batch_idx + 1) % 10 == 0:
            print(f"[Train] Batch {batch_idx+1}/{len(loader)} - Loss: {total_batch_loss.item():.4f}")
    
    avg_loss = total_loss / len(loader)
    print(f"[Train] Epoch complete. Average loss: {avg_loss:.4f}")
    return avg_loss

def validate(model, data_list, device, num_cell_types, alpha=1.0):
    model.eval()
    total_loss = 0
    
    loader = DataLoader(data_list, batch_size=1, shuffle=False, collate_fn=pyg_collate_fn)
    criterion_state = nn.KLDivLoss(reduction='batchmean')

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            batch = batch.to(device)
            
            state_pred, area_pred, perim_pred, adj_pred = model(batch.x, batch.edge_index, batch.edge_attr)

            loss_adj   = distance_based_adj_loss(adj_pred, batch.next_adj, batch, alpha=alpha)
            loss_area  = distance_based_node_prop_loss(area_pred, batch.next_area.unsqueeze(-1), batch, alpha=alpha)
            loss_perim = distance_based_node_prop_loss(perim_pred, batch.next_perim.unsqueeze(-1), batch, alpha=alpha)
            loss_state = criterion_state(state_pred, batch.next_state)

            batch_loss = loss_adj + loss_area + loss_perim + loss_state
            total_loss += batch_loss.item()
            
            if (batch_idx + 1) % 10 == 0:
                print(f"[Val] Batch {batch_idx+1}/{len(loader)} - Cumulative Loss: {total_loss:.4f}")
    
    avg_loss = total_loss / len(loader)
    print(f"[Val] Validation complete. Average loss: {avg_loss:.4f}")
    return avg_loss

##########################################################################
# 6. K-Fold across Directories
##########################################################################

def k_fold_training(
    data_dirs, param_files, timepoints, timepoint_interval,
    node_dim, edge_dim, hidden_dim, num_cell_types,
    epochs=100, lr=1e-3, k_folds=5, alpha=1.0
):
    """
    We now separate each directory's data as its own timeseries chunk,
    so we can do K-Fold at the directory level rather than merging all samples.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # 1) Build a list-of-lists: all_data_by_dir[d] = list of Data for directory d
    all_data_by_dir = []  # shape: [num_directories][some_number_of_timepoints]
    
    print("Preprocessing data (keeping each directory separate)...")
    for dir_idx, (data_dir, param_file) in enumerate(zip(data_dirs, param_files)):
        print(f"Processing directory {data_dir} with parameter file {param_file}...")
        params = parse_parameters(param_file)
        
        dir_data = []  # store all (current->next) pairs for this directory
        for t in range(0, timepoints - timepoint_interval, timepoint_interval):
            print(f"Loading timepoint {t} and {t + timepoint_interval} for dir {dir_idx}...")
            current = load_matrices(data_dir, t)
            next_t  = load_matrices(data_dir, t + timepoint_interval)
            
            g_current = initialize_gca(*current, params)
            g_next    = initialize_gca(*next_t, params)
            
            data = from_networkx(g_current)

            # node features
            node_feat = []
            num_nodes = g_current.number_of_nodes()
            for i in g_current.nodes:
                area = g_current.nodes[i]["area"]
                perim = g_current.nodes[i]["perimeter"]
                st   = g_current.nodes[i]["state"]
                feats = np.concatenate([[area, perim], st])
                node_feat.append(feats)
            node_feat = np.array(node_feat, dtype=np.float32)
            
            # optional "contrastive" feature steps...
            type1_nodes = 0
            for i in range(num_nodes):
                st_idx = 2
                st_vector = node_feat[i, st_idx:(st_idx+num_cell_types)]
                node_type = np.argmax(st_vector)
                if node_type == 1:
                    type1_nodes += 1
            global_type1_fraction = float(type1_nodes) / float(num_nodes) if num_nodes > 0 else 0.0
            
            local_fractions = np.zeros(num_nodes, dtype=np.float32)
            for i in range(num_nodes):
                neighbors = list(g_current.neighbors(i))
                if len(neighbors) == 0:
                    local_fractions[i] = 0.0
                    continue
                n_type1 = 0
                for nbr in neighbors:
                    st_idx = 2
                    st_vector = node_feat[nbr, st_idx:(st_idx+num_cell_types)]
                    nbr_type = np.argmax(st_vector)
                    if nbr_type == 1:
                        n_type1 += 1
                local_fractions[i] = n_type1 / float(len(neighbors))
            rel_diff = local_fractions - global_type1_fraction
            node_feat = np.hstack([node_feat, rel_diff.reshape(-1,1)])
            
            data.x = torch.tensor(node_feat, dtype=torch.float)
            
            # edge features
            edge_feat = []
            for (u, v) in zip(data.edge_index[0], data.edge_index[1]):
                edge = g_current.edges[int(u), int(v)]
                edge_feat.append([
                    edge["adhesion"],
                    edge["repulsion_radius"],
                    edge["repulsion_coefficient"]
                ])
            data.edge_attr = torch.tensor(edge_feat, dtype=torch.float)
            
            # targets
            data.next_state = torch.tensor(
                [g_next.nodes[i]["state"] for i in g_current.nodes],
                dtype=torch.float
            )
            data.next_area = torch.tensor(
                [g_next.nodes[i]["area"] for i in g_current.nodes],
                dtype=torch.float
            )
            data.next_perim = torch.tensor(
                [g_next.nodes[i]["perimeter"] for i in g_current.nodes],
                dtype=torch.float
            )
            data.next_adj = torch.tensor(
                nx.to_numpy_array(g_next)[data.edge_index[0], data.edge_index[1]],
                dtype=torch.float
            )

            dir_data.append(data)
        
        all_data_by_dir.append(dir_data)
    
    num_dirs = len(all_data_by_dir)
    print(f"Number of directories: {num_dirs}")
    
    # 2) K-Fold across the directories themselves
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    best_model = None
    best_loss = float('inf')
    
    # We'll do an index over [0..num_dirs-1] for folds
    dir_indices = list(range(num_dirs))
    
    for fold, (train_dir_idx, val_dir_idx) in enumerate(kf.split(dir_indices)):
        print(f"\n--- Starting Fold {fold+1}/{k_folds} ---")
        
        # gather training data from all directories in train_dir_idx
        train_data = []
        for d_idx in train_dir_idx:
            train_data.extend(all_data_by_dir[d_idx])  # add that entire timeseries
        
        # gather validation data
        val_data = []
        for d_idx in val_dir_idx:
            val_data.extend(all_data_by_dir[d_idx])
        
        print(f"  Using directories {train_dir_idx} for training, {val_dir_idx} for validation.")
        print(f"  Training data size: {len(train_data)}")
        print(f"  Validation data size: {len(val_data)}")
        
        # check node_dim from the first sample in train_data
        actual_node_dim = train_data[0].x.shape[1]
        print(f"  Detected node_dim = {actual_node_dim}")
        
        model = DeepGraphPredictor(actual_node_dim, edge_dim, hidden_dim, num_cell_types).to(device)
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
        
        best_val_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(epochs):
            print(f"\nEpoch {epoch+1}/{epochs}")
            train_loss = train_epoch(model, train_data, optimizer, device, num_cell_types, alpha=alpha)
            val_loss   = validate(model, val_data, device, num_cell_types, alpha=alpha)
            scheduler.step(val_loss)
            
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(model.state_dict(), f"best_fold{fold}.pth")
                print(f"    --> New best model saved for fold {fold+1} with validation loss: {best_val_loss:.4f}")
            else:
                patience_counter += 1
                if patience_counter >= 10:
                    print("    --> Early stopping triggered.")
                    break
        
        if best_val_loss < best_loss:
            best_loss = best_val_loss
            best_model = model
    
    print(f"\nTraining complete. Best validation loss across folds: {best_loss:.4f}")
    
    if best_model is not None:
        torch.save(best_model.state_dict(), "best_model.pth")
        print("Best model saved to 'best_model.pth'")
    else:
        print("No best model found.")
    
    return best_model

##########################################################################
# 7. Main Execution
##########################################################################

if __name__ == "__main__":
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/{i}_matrix_output"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]

    timepoints = 2000
    timepoint_interval = 100
    
    num_cell_types = 2

    # Our original code had node_dim = 2 + num_cell_types,
    # but now we are adding +1 for the "contrastive" relative feature:
    # We can keep node_dim the same and compute it dynamically later, or just
    # set node_dim = 2 + num_cell_types + 1 = 2 + 2 + 1 = 5
    node_dim = 2 + num_cell_types  # We'll adjust it dynamically in the code above
    edge_dim = 3
    hidden_dim = 64

    best_model = k_fold_training(
        data_dirs, param_files,
        timepoints, timepoint_interval,
        node_dim=node_dim,  # We'll let the code override
        edge_dim=edge_dim,
        hidden_dim=hidden_dim,
        num_cell_types=num_cell_types,
        epochs=15, lr=1e-4, k_folds=3,
        alpha=4
    )
    print("Done.")


Using device: cpu
Preprocessing data (keeping each directory separate)...
Processing directory /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/1_matrix_output with parameter file /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py...
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Successfully parsed parameters: {'domain_size': [60, 14], 'init_noise': 0.005, 'rng_seed': 1, 'dt': 0.25, 'tMax': 501, 'stripe_thickness': 4, 'stripe_density': 0.5, 'v0': [0.1, 1.3], 'W': array([[0.  , 0.08],
       [0.08, 0.  ]]), 'A0': [0.9, 0.9], 'P0': [3.812, 3.812], 'Dr': 50, 'kappa_A': 0.4, 'kappa_P': 0.07, 'a': 0.25, 'k': 2.5}
Loading timepoint 0 and 100 for dir 0...
Loading matrices for timepoint 0 from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/1_matrix_output
Loading matrices for timepoint 100 from /Users/sophia01px2019/Downl

In [132]:
import os
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

import torch
import torch.nn.functional as F
from torch_geometric.utils import from_networkx
from torch_geometric.data import Data

# REUSE your training-time helpers (simplified here):
# ---------------------------------------------------
# parse_parameters(param_path)
# initialize_gca(graph_mat, properties_mat, state_mat, params)
# class GraphPredictor(nn.Module): ...


def build_pyg_data_from_gca(graph_mat, properties_mat, state_mat, params, initialize_gca_func):
    """
    Builds a PyG Data object from your GCA initialization logic,
    mirroring exactly how you did it in training.
    """
    # This uses your existing initialize_gca(...) to get a NetworkX graph with attributes
    g = initialize_gca_func(graph_mat, properties_mat, state_mat, params)
    data = from_networkx(g)

    num_nodes = g.number_of_nodes()

    # 1) Compute global fraction of type 1
    type1_count = 0
    for i in g.nodes:
        st = g.nodes[i]["state"]
        if np.argmax(st) == 1:
            type1_count += 1
    global_type1_fraction = type1_count / float(num_nodes) if num_nodes > 0 else 0.0

    # 2) For each node, compute local fraction of type 1 neighbors
    local_fractions = np.zeros(num_nodes, dtype=np.float32)
    for i in range(num_nodes):
        neighbors = list(g.neighbors(i))
        if len(neighbors) == 0:
            local_fractions[i] = 0.0
            continue
        local_count = 0
        for nbr in neighbors:
            stn = g.nodes[nbr]["state"]
            if np.argmax(stn) == 1:
                local_count += 1
        local_fractions[i] = local_count / float(len(neighbors))

    node_features = []
    for i in range(num_nodes):
        area_i = g.nodes[i]["area"]
        perim_i = g.nodes[i]["perimeter"]
        state_i = g.nodes[i]["state"]  # shape [num_cell_types]

        # 3) rel_diff for node i
        rel_diff_i = local_fractions[i] - global_type1_fraction

        # 4) now append rel_diff to the original features
        feats = np.concatenate([[area_i, perim_i], state_i, [rel_diff_i]])
        node_features.append(feats)



    data.x = torch.tensor(node_features, dtype=torch.float)

    # Edge features: (adhesion, repulsion_radius, repulsion_coefficient)
    edge_features = []
    for (u, v) in zip(data.edge_index[0], data.edge_index[1]):
        edge_attr = g.edges[int(u), int(v)]
        feats = [
            edge_attr["adhesion"],
            edge_attr["repulsion_radius"],
            edge_attr["repulsion_coefficient"]
        ]
        edge_features.append(feats)
    data.edge_attr = torch.tensor(edge_features, dtype=torch.float)

    return data

# ---------------------------------------------------
# VISUALIZATION HELPERS
# ---------------------------------------------------

def color_code_adjacency_matrix(graph_mat, node_types):
    """
    Color-codes the adjacency matrix based on node types.
    node_types: array of shape [N], each value is 0 or 1 for the cell type.
    """
    # We'll create a matrix with integer codes:
    #   0 = No edge
    #   1 = Edge between same-type=0
    #   2 = Edge between same-type=1
    #   3 = Edge between type=0 and type=1
    color_coded_mat = np.zeros_like(graph_mat, dtype=int)

    n = graph_mat.shape[0]
    for i in range(n):
        for j in range(n):
            if graph_mat[i, j] == 1:
                # There's an edge
                if node_types[i] == node_types[j]:
                    if node_types[i] == 0:
                        color_coded_mat[i, j] = 1  # same type=0
                    else:
                        color_coded_mat[i, j] = 2  # same type=1
                else:
                    color_coded_mat[i, j] = 3  # different type
    return color_coded_mat

def plot_adjacency_matrix(graph_mat, state_mat, save_path):
    """
    Saves a color-coded adjacency matrix plot.
    Expects state_mat to be either shape [N, 2] with probabilities or
    shape [N] with discrete cell types.
    """
    # If state_mat is a distribution, get the argmax
    if len(state_mat.shape) == 2 and state_mat.shape[1] > 1:
        node_types = np.argmax(state_mat, axis=1)
    else:
        node_types = state_mat.flatten().astype(int)

    color_coded = color_code_adjacency_matrix(graph_mat, node_types)
    
    # Define a colormap:
    # index 0=white, 1=orange, 2=purple, 3=blue
    cmap = ListedColormap(["white", "#FFA500", "#800080", "#0000FF"])
    
    plt.figure(figsize=(8, 8))
    plt.title("Color-Coded Adjacency Matrix")
    plt.imshow(color_coded, cmap=cmap, interpolation="nearest")
    cbar = plt.colorbar(ticks=[0, 1, 2, 3])
    cbar.ax.set_yticklabels([
        "No Edge", 
        "Same Type 0 (Orange)", 
        "Same Type 1 (Purple)",
        "Diff Types (Blue)"
    ])
    plt.xlabel("Node Index")
    plt.ylabel("Node Index")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_graph_representation(graph_mat, state_mat, save_path, pos=None):
    """
    Visualizes the adjacency graph with node colors (and optional alpha).
    Returns 'pos' so you can keep a consistent layout across frames if desired.
    """
    # Build a NetworkX graph
    g = nx.from_numpy_array(graph_mat, create_using=nx.Graph)

    # Convert state_mat to node types
    # If shape is [N,2], interpret as distribution => argmax
    if len(state_mat.shape) == 2 and state_mat.shape[1] > 1:
        node_types = np.argmax(state_mat, axis=1)
    else:
        node_types = state_mat.flatten().astype(int)

    # Pick color & alpha by type
    node_colors = []
    node_alphas = []
    for t in node_types:
        if t == 0:
            node_colors.append("yellow")
            node_alphas.append(0.5)
        else:
            node_colors.append("purple")
            node_alphas.append(1.0)
    
    # Layout
    if pos is None:
        pos = nx.kamada_kawai_layout(g)

    fig, ax = plt.subplots(figsize=(8,8))
    nx.draw_networkx_edges(g, pos, edge_color="gray", ax=ax)
    nx.draw_networkx_nodes(
        g, pos, 
        node_color=node_colors, alpha=node_alphas,
        node_size=50, ax=ax
    )

    # Add a legend
    patches = [
        mpatches.Patch(color="yellow", label="Cell Type 0 (alpha=0.5)"),
        mpatches.Patch(color="purple", label="Cell Type 1 (alpha=1.0)")
    ]
    ax.legend(handles=patches, loc="upper right", title="Cell Types", fontsize="small")
    plt.title("Predicted Graph Representation")
    
    plt.savefig(save_path)
    plt.close()
    return pos

def plot_area_perimeter_bar(properties_mat, state_mat, save_path):
    """
    Plots bar charts of area & perimeter, colored by cell type.
    Expects properties_mat to be shape [N, 2], with columns= [Area, Perimeter].
    """
    if len(state_mat.shape) == 2 and state_mat.shape[1] > 1:
        node_types = np.argmax(state_mat, axis=1)
    else:
        node_types = state_mat.flatten().astype(int)

    # Color each bar by cell type
    node_colors = []
    for t in node_types:
        if t == 0:
            node_colors.append("yellow")
        else:
            node_colors.append("purple")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    # Area
    axes[0].bar(range(len(properties_mat)), properties_mat[:,0], color=node_colors)
    axes[0].set_title("Predicted Cell Area")
    axes[0].set_xlabel("Node Index")
    axes[0].set_ylabel("Area")

    # Perimeter
    axes[1].bar(range(len(properties_mat)), properties_mat[:,1], color=node_colors)
    axes[1].set_title("Predicted Cell Perimeter")
    axes[1].set_xlabel("Node Index")
    axes[1].set_ylabel("Perimeter")

    # Legend
    patches = [
        mpatches.Patch(color="yellow", label="Cell Type 0"),
        mpatches.Patch(color="purple", label="Cell Type 1")
    ]
    fig.legend(handles=patches, loc="center right", title="Cell Types", fontsize="small")
    plt.tight_layout(rect=[0,0,0.85,1])
    
    plt.savefig(save_path)
    plt.close()

def visualize_prediction(output_dir, timepoint, graph_mat, state_mat, properties_mat):
    """
    Saves three plots for the given timepoint:
     1) Color-coded adjacency matrix
     2) Node-link diagram (graph representation)
     3) Bar plots of area & perimeter
    """
    # Create subfolders
    adjacency_dir = os.path.join(output_dir, "adjacency_matrix")
    graph_dir = os.path.join(output_dir, "graph_representation")
    state_dir = os.path.join(output_dir, "states")
    os.makedirs(adjacency_dir, exist_ok=True)
    os.makedirs(graph_dir, exist_ok=True)
    os.makedirs(state_dir, exist_ok=True)

    # 1) Adjacency matrix
    adj_save_path = os.path.join(adjacency_dir, f"{timepoint}.png")
    plot_adjacency_matrix(graph_mat, state_mat, adj_save_path)

    # 2) Graph representation
    graph_save_path = os.path.join(graph_dir, f"{timepoint}.png")
    plot_graph_representation(graph_mat, state_mat, graph_save_path)

    # 3) Area & Perimeter bar plots
    state_save_path = os.path.join(state_dir, f"{timepoint}.png")
    plot_area_perimeter_bar(properties_mat, state_mat, state_save_path)

    print(f"  -> Visualization saved for timepoint {timepoint}.")


# ---------------------------------------------------
# MAIN SEQUENTIAL PREDICTION + VISUALIZATION
# ---------------------------------------------------

def sequential_prediction_with_visuals(
    data_dir,
    param_file,
    model_path,
    start_timepoint,
    timepoints,
    node_dim, edge_dim, hidden_dim, num_cell_types,
    initialize_gca_func,  # pass your "initialize_gca" here
    parse_params_func,     # pass your "parse_parameters" here
    DeepGraphPredictorClass,   # pass the class definition
    output_dir
):
    """
    Performs the same sequential prediction as before, but also generates & saves visualizations.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # 1. Load the trained model
    print(f"Loading model from {model_path}")
    model = DeepGraphPredictorClass(node_dim, edge_dim, hidden_dim, num_cell_types)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    # 2. Parse parameters + load the initial .npy
    params = parse_params_func(param_file)

    # For the starting timepoint:
    graph_mat = np.load(os.path.join(data_dir, f"{start_timepoint}_graph_mat.npy"))
    state_mat = np.load(os.path.join(data_dir, f"{start_timepoint}_state_mat.npy"))
    properties_mat = np.load(os.path.join(data_dir, f"{start_timepoint}_properties_mat.npy"))

    # Create output_dir if not exists
    os.makedirs(output_dir, exist_ok=True)

    # Save them immediately so we have the baseline
    np.save(os.path.join(output_dir, f"{start_timepoint}_graph_mat.npy"), graph_mat)
    np.save(os.path.join(output_dir, f"{start_timepoint}_state_mat.npy"), state_mat)
    np.save(os.path.join(output_dir, f"{start_timepoint}_properties_mat.npy"), properties_mat)
    print(f"Initial (time={start_timepoint}) saved to {output_dir}.")
    visualize_prediction(output_dir, start_timepoint, graph_mat, state_mat, properties_mat)

    # 3. Iterate over timepoints
    for idx in range(1, len(timepoints)):
        prev_t = timepoints[idx - 1]
        curr_t = timepoints[idx]
        print(f"\n--- Predicting from time {prev_t} to {curr_t} ---")

        # Build PyG data from the *previous* state
        data_pyg = build_pyg_data_from_gca(
            graph_mat, properties_mat, state_mat, params, initialize_gca_func
        )
        data_pyg = data_pyg.to(device)

        # Forward pass
        with torch.no_grad():
            state_pred, area_pred, perim_pred, adj_pred = model(
                data_pyg.x, data_pyg.edge_index, data_pyg.edge_attr
            )

        # Convert predictions back to numpy
        state_pred = state_pred.exp().cpu().numpy()  # was log-softmax -> exp
        area_pred  = area_pred.cpu().numpy().reshape(-1)
        perim_pred = perim_pred.cpu().numpy().reshape(-1)
        adj_logits = adj_pred.cpu().numpy()  # shape [num_edges, 2]
        adj_class  = np.argmax(adj_logits, axis=1)  # shape [num_edges], 0 or 1
        row = data_pyg.edge_index[0].cpu().numpy()
        col = data_pyg.edge_index[1].cpu().numpy()


        edge_index = data_pyg.edge_index.cpu().numpy()  # shape [2, E]

        # Sigmoid on adjacency logits
        adj_probs = 1.0 / (1.0 + np.exp(-adj_logits))  # shape [E]

        # Threshold at 0.5
        num_nodes = state_pred.shape[0]
        new_graph_mat = np.zeros((num_nodes, num_nodes), dtype=np.float32)
        for e_idx in range(edge_index.shape[1]):
            u = edge_index[0, e_idx]
            v = edge_index[1, e_idx]
            if adj_probs[e_idx] >= 0.05:
                new_graph_mat[u, v] = 1.0
                new_graph_mat[v, u] = 1.0

        # Build updated arrays
        new_state_mat = state_pred   # shape [N, num_cell_types]
        # Or if you prefer discrete: new_state_mat = np.argmax(state_pred, axis=1)
        new_properties_mat = np.stack([area_pred, perim_pred], axis=1)  # shape [N, 2]

        # Save as .npy
        np.save(os.path.join(output_dir, f"{curr_t}_graph_mat.npy"), new_graph_mat)
        np.save(os.path.join(output_dir, f"{curr_t}_state_mat.npy"), new_state_mat)
        np.save(os.path.join(output_dir, f"{curr_t}_properties_mat.npy"), new_properties_mat)

        # Also generate & save graphs
        visualize_prediction(output_dir, curr_t, new_graph_mat, new_state_mat, new_properties_mat)

        # Update for next iteration
        graph_mat = new_graph_mat
        state_mat = new_state_mat
        properties_mat = new_properties_mat

    print("\nSequential prediction + visualization complete.")


# ---------------------------------------------------
# EXAMPLE STANDALONE USAGE
# ---------------------------------------------------
if __name__ == "__main__":
    # Suppose you have the following from your training phase:
    #
    # from train_script import (
    #    parse_parameters,       # param_file -> dict of {W, v0, Dr, kappa_A, etc.}
    #    initialize_gca,        # graph_mat, properties_mat, state_mat, params -> Nx Graph
    #    GraphPredictor         # your GINEConv-based model class
    # )
    #
    # We'll just assume they're available in the namespace.

    data_dir = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/matrix_output_1_test"
    param_file = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py"
    output_dir = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/predictions"

    model_path = "best_model.pth"  # your trained model

    start_timepoint = 0
    timepoints = list(range(0, 2001, 100))  # 0, 100, 200, ... 2000

    num_cell_types = 2
    node_dim = 2 + num_cell_types +1  # (area, perimeter) + state distribution
    edge_dim = 3
    hidden_dim = 64

    # Run the sequential prediction + plotting
    sequential_prediction_with_visuals(
        data_dir=data_dir,
        param_file=param_file,
        model_path=model_path,
        start_timepoint=start_timepoint,
        timepoints=timepoints,
        node_dim=node_dim,
        edge_dim=edge_dim,
        hidden_dim=hidden_dim,
        num_cell_types=num_cell_types,
        initialize_gca_func=initialize_gca,
        parse_params_func=parse_parameters,
        DeepGraphPredictorClass=DeepGraphPredictor,
        output_dir=output_dir
    )


Using device: cpu
Loading model from best_model.pth
Parsing parameters from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Successfully parsed parameters: {'domain_size': [60, 14], 'init_noise': 0.005, 'rng_seed': 1, 'dt': 0.25, 'tMax': 501, 'stripe_thickness': 4, 'stripe_density': 0.5, 'v0': [0.1, 1.3], 'W': array([[0.  , 0.08],
       [0.08, 0.  ]]), 'A0': [0.9, 0.9], 'P0': [3.812, 3.812], 'Dr': 50, 'kappa_A': 0.4, 'kappa_P': 0.07, 'a': 0.25, 'k': 2.5}
Initial (time=0) saved to /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/predictions.
  -> Visualization saved for timepoint 0.

--- Predicting from time 0 to 100 ---
Initializing GCA graph...
  -> Visualization saved for timepoint 100.

--- Predicting from time 100 to 200 ---
Initializing GCA graph...
  -> Visualization saved for timepoint 200.

--- Predicting from time 200 to 300 ---
Initializing GCA graph...
  -> Visualization saved for timepoint 300.

--- Predicting from tim

KeyboardInterrupt: 